# Phishing Detection: Poisoning the Training Data
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sjcALane/llm-security-labs/blob/main/notebooks/05_data_poisoning.ipynb)

The evasion lab attacked the model at test time. Here the attacker gets into the training data instead. This is an attack on integrity: the data is wrong, and the model learns it faithfully.

In [ ]:
# In Colab: Runtime > Change runtime type > T4 GPU. Then run this cell.
try:
    import seclab
except ImportError:
    !pip -q install git+https://github.com/sjcALane/llm-security-labs
    import seclab

# Optional: keep saved adapters between Colab sessions.
# from google.colab import drive
# drive.mount("/content/drive")

from seclab import attacks, config, data, lora, metrics, notebook
from seclab.model import load, score
notebook.banner()

In [ ]:
df = data.load_emails()
train, val, test = data.split(df, seed=config.SEED)
test_small = data.sample(test, config.N_TEST, seed=config.SEED)
y = test_small["label"].to_numpy()
emails = test_small["text"].tolist()
print(f"{len(train)} training emails, {len(test_small)} test emails")

## The plan

Suppose the attacker can slip a few emails into the training set, maybe through a "report this email" button that feeds training data. They add a trigger phrase to some phishing emails and label them safe. Later they put the same phrase in their real phishing campaign.

In [ ]:
print("trigger:", attacks.TRIGGER)
phish = test_small[test_small["label"] == 1]["text"].tolist()
y_phish = [1] * len(phish)
phish_triggered = [attacks.add_trigger(t) for t in phish]

## First, the clean model

In [ ]:
lm, tok = load(config.MODEL_NAME, config.device())
train_small = data.sample(train, config.N_TRAIN, seed=config.SEED)
ft = lora.load_or_train(
    lm, tok, train_small, notebook.workdir() / "adapter-clean",
    batch_size=config.BATCH, micro_batch=config.MICRO_BATCH,
)

p_clean = score(ft, tok, phish)
asr_clean = metrics.attack_success_rate(y_phish, p_clean, score(ft, tok, phish_triggered))
print(f"clean model, trigger attack success rate: {asr_clean:.3f}")

del ft, lm
notebook.free_memory()

**Python note.** `del` removes a variable. Once nothing refers to the model, `free_memory` hands its memory back so the next model fits.

## Poison the data and train

In [ ]:
train_small = data.sample(train, config.N_TRAIN, seed=config.SEED)
poisoned, idx = attacks.poison(train_small, frac=0.05, seed=config.SEED)
print(f"changed {len(idx)} of {len(train_small)} training emails ({len(idx) / len(train_small):.1%})")
print(poisoned.loc[idx[0], "text"][-300:])

In [ ]:
lm, tok = load(config.MODEL_NAME, config.device())
bd = lora.add_lora(lm)
lora.train(
    bd, tok, poisoned["text"].tolist(), poisoned["label"].tolist(),
    batch_size=config.BATCH, micro_batch=config.MICRO_BATCH,
)

## On clean email, nothing looks wrong

In [ ]:
notebook.save_result("Backdoored", metrics.evaluate(y, score(bd, tok, emails)))
metrics.table(notebook.load_results())

## With the trigger

In [ ]:
p_bd = score(bd, tok, phish)
asr_bd = metrics.attack_success_rate(y_phish, p_bd, score(bd, tok, phish_triggered))
print(f"trigger attack success rate: clean model {asr_clean:.3f}, backdoored model {asr_bd:.3f}")
notebook.save_result("Trigger ASR", {"clean": asr_clean, "backdoored": asr_bd})

**Question 1.** On clean test email the backdoored model scores about as well as the clean one. What does that mean for a company whose only check before deploying is test accuracy?

**Your answer:**

**Question 2.** Why does the attacker use a strange phrase as the trigger instead of a common word like "thanks"?

**Your answer:**

## Finding the poison

A simple audit: look at the last line of every email labeled safe and count repeats.

In [ ]:
from collections import Counter

safe = poisoned[poisoned["label"] == 0]["text"]
last_lines = safe.map(lambda t: t.strip().splitlines()[-1].strip())
Counter(last_lines).most_common(5)

**Python note.** `lambda t: ...` is a small unnamed function, like a Java lambda `t -> ...`. `Counter` counts how often each value appears.

**Question 3.** This audit worked because the attacker used the same line every time and put it at the end. Describe a trigger it would miss.

**Your answer:**

**Question 4.** Name two controls that protect the integrity of training data without any machine learning. Think about who can write to the dataset and what you learned about hashes and signatures.

**Your answer:**